# FEDformer Native Adapter v1

目标：把 `fedformer_mse` 从 proxy feature adapter 推进到 `src/model` 的 native sequence adapter。Notebook 只保留实验流程和中文结论；可复用训练、回测、analysis 逻辑都调用 `src/`。外部论文和官方仓库只保存 URL 与自有摘要，不复制全文或第三方代码。

In [1]:
from pathlib import Path
import json
import sys

REPO_ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / 'AGENTS.md').exists() and (path / 'src').exists())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.data import build_supervised_dataset, load_soxl_support_frame
from src.model.capabilities import model_capability_table, model_fidelity_summary
from src.model.matrix import run_quantbench_matrix_smoke
from src.model.sequence import FEDformerFrequencyRegressor
from src.model.training import fit_estimator_for_model

OUTPUT_DIR = REPO_ROOT / 'research' / 'fedformer_native_adapter_v1'
CARD_DIR = OUTPUT_DIR / 'matrix_card'

outputs = run_quantbench_matrix_smoke(
    card_dir=CARD_DIR,
    start='2026-01-01',
    end='2026-06-09',
    train_end='2026-04-30',
    model_names=('ridge_mse', 'dlinear_mse', 'patch_tst_mse', 'autoformer_mse', 'fedformer_mse'),
)

frame = load_soxl_support_frame(start='2026-01-01', end='2026-06-09')
dataset = build_supervised_dataset(frame, horizon=1)
train = dataset.frame.loc[dataset.frame['date'] <= '2026-04-30'].copy()
estimator, constant_prediction = fit_estimator_for_model(
    'fedformer_mse',
    train=train,
    feature_columns=dataset.feature_columns[:5],
)
assert constant_prediction is None
assert isinstance(estimator, FEDformerFrequencyRegressor)
assert hasattr(estimator, 'trend_model_')
assert hasattr(estimator, 'frequency_model_')
assert estimator.frequency_modes_ == (1, 2)

capability = model_capability_table(('fedformer_mse',)).iloc[0].to_dict()
fidelity = model_fidelity_summary(('ridge_mse', 'dlinear_mse', 'patch_tst_mse', 'autoformer_mse', 'fedformer_mse'))
fedformer_row = outputs.model_comparison.loc[outputs.model_comparison['model'] == 'fedformer_mse'].iloc[0].to_dict()
analysis_summary = json.loads((CARD_DIR / 'analysis_summary.json').read_text(encoding='utf-8'))
notebook_summary = {
    'schema_version': 'research.fedformer_native_adapter_v1',
    'status': 'completed',
    'paper_reference': {
        'source_id': 'arxiv:2201.12740',
        'url': 'https://arxiv.org/abs/2201.12740',
        'official_repo': 'https://github.com/MAZiqing/FEDformer',
        'usage': '自有摘要和结构假设；不复制论文全文或第三方代码。',
    },
    'model_names': ['ridge_mse', 'dlinear_mse', 'patch_tst_mse', 'autoformer_mse', 'fedformer_mse'],
    'fedformer_estimator': type(estimator).__name__,
    'fedformer_has_trend_head': hasattr(estimator, 'trend_model_'),
    'fedformer_has_frequency_head': hasattr(estimator, 'frequency_model_'),
    'fedformer_frequency_modes': list(estimator.frequency_modes_),
    'fedformer_top_k_modes': estimator.top_k_modes,
    'constant_prediction': constant_prediction,
    'fedformer_paper_fidelity': capability['paper_fidelity'],
    'fedformer_is_proxy': bool(capability['is_proxy']),
    'model_fidelity': fidelity,
    'top_model': str(outputs.model_comparison.iloc[0]['model']),
    'fedformer_return_pct': float(fedformer_row['return_pct']),
    'fedformer_relative_return_pct': float(fedformer_row['relative_return_pct']),
    'fedformer_max_drawdown_pct': float(fedformer_row['max_drawdown_pct']),
    'fedformer_sharpe': float(fedformer_row['sharpe']),
    'analysis_summary': analysis_summary,
    'artifact_links': {
        'matrix_card': 'matrix_card/',
        'model_comparison': 'matrix_card/model_comparison.csv',
        'model_run_ledger': 'matrix_card/model_run_ledger.csv',
        'model_capabilities': 'matrix_card/model_capabilities.csv',
        'analysis_manifest': 'matrix_card/analysis_manifest.json',
        'artifact_manifest': 'matrix_card/artifact_manifest.json',
    },
}
(OUTPUT_DIR / 'notebook_summary.json').write_text(
    json.dumps(notebook_summary, ensure_ascii=False, indent=2),
    encoding='utf-8',
)
notebook_summary


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


{'schema_version': 'research.fedformer_native_adapter_v1',
 'status': 'completed',
 'paper_reference': {'source_id': 'arxiv:2201.12740',
  'url': 'https://arxiv.org/abs/2201.12740',
  'official_repo': 'https://github.com/MAZiqing/FEDformer',
  'usage': '自有摘要和结构假设；不复制论文全文或第三方代码。'},
 'model_names': ['ridge_mse',
  'dlinear_mse',
  'patch_tst_mse',
  'autoformer_mse',
  'fedformer_mse'],
 'fedformer_estimator': 'FEDformerFrequencyRegressor',
 'fedformer_has_trend_head': True,
 'fedformer_has_frequency_head': True,
 'fedformer_frequency_modes': [1, 2],
 'fedformer_top_k_modes': 2,
 'constant_prediction': None,
 'fedformer_paper_fidelity': 'native_sequence',
 'fedformer_is_proxy': False,
 'model_fidelity': {'schema_version': 'model.fidelity.v1',
  'model_count': 5,
  'fidelity_counts': {'native_library': 1, 'native_sequence': 4},
  'proxy_models': [],
  'native_models': ['autoformer_mse',
   'dlinear_mse',
   'fedformer_mse',
   'patch_tst_mse',
   'ridge_mse'],
  'promotion_blocker_count':

## 中文结论

- 做了什么：在 `src/model.sequence` 增加 `FEDformerFrequencyRegressor`，把 `fedformer_mse` 从 `proxy_feature_adapter` 升级为 `native_sequence`。实现保留 FEDformer 的核心结构假设：移动平均分解 trend/residual，并从 residual 中抽取稀疏 Fourier mode、频域幅度和重构统计，再用 trend/frequency 两个 head 拟合。
- 核心指标：本 notebook 训练并回测 `ridge_mse`、`dlinear_mse`、`patch_tst_mse`、`autoformer_mse`、`fedformer_mse`，输出 `model_comparison.csv`、`daily_backtest.csv`、`model_run_ledger.csv`、`qlib_risk_metrics.csv`、`model_effect_deltas.csv` 和 manifest。具体收益、回撤、Sharpe 以 `notebook_summary.json` 与 `matrix_card/analysis_summary.json` 为准。
- regime 差异：本轮复用 `src/analysis` 的 window/regime、qlib risk、model effect delta；FEDformer 是否有价值不能只看全区间收益，要重点看 stress/downtrend 下的相对收益、回撤、稳定性和换手。
- 是否晋升：否。`native_sequence` 只表示实现形态已从 proxy 推进为稳定训练接口；production 晋升仍需要更长历史窗口、更多市场 regime、成本压力和 paper/sim replay 复验。
- 风险：当前是紧凑实现，不等价于完整论文训练规模；SOXL 单标的样本短，稀疏频域 mode 可能过拟合近期周期；外部论文和官方仓库仅作为结构参考，未复制第三方代码或全文。
- 下一步：刷新全模型 ledger，使 native/proxy 计数同步；继续验证 `rsr_mse`、`thgnn_mse`、`sthan_mse` 等剩余 proxy replacement。